<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/07_GES_Aware_Genomic_RAG_Cell_7C0_Embedding_and_Semantic_Retrieval_Execution_FINAL_CORRECTED_V6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Google Drive and install the exact Cell 7B4 frozen execution packages.
from pathlib import Path
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; Google Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root does not exist: {ROOT}\n'
        'Confirm that Google Drive is mounted and the project folder name is exact.'
    )

EXACT_REQUIREMENTS = [
    'sentence-transformers==5.6.0',
    'transformers==5.14.1',
    'faiss-cpu==1.14.3',
    'openai==2.45.0',
]

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--disable-pip-version-check',
    '--quiet',
    *EXACT_REQUIREMENTS,
])

print(f'Project root: {ROOT}')
print('Exact frozen execution packages installed or verified.')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study
Exact frozen execution packages installed or verified.


## 1. Imports, deterministic controls, frozen identities, and output locations

In [2]:
from __future__ import annotations

import csv
import gc
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import platform
import random
import re
import shutil
import tempfile
import time
import unicodedata
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

# These environment controls are set before importing torch/transformers/faiss.
os.environ['PYTHONHASHSEED'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

import faiss
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from huggingface_hub import HfApi, snapshot_download
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

CELL_ID = '7C0'
STAGE = '7C'
PACKAGE_VERSION = '1.0.0'
PACKAGE_NAME = 'cell_7c0_embedding_and_semantic_retrieval_v1'
NOTEBOOK_NAME = '07_GES_Aware_Genomic_RAG_Cell_7C0_Embedding_and_Semantic_Retrieval_Execution_FINAL_CORRECTED_V6.ipynb'
CREATED_UTC = datetime.now(timezone.utc).isoformat()
FROZEN_SEED = 20260722

EXPECTED_CELL_7B5_DECISION = (
    'PASS_STAGE7B5_CELL7B4_CONFIGURATION_AND_SCORE_BLIND_EXECUTION_INPUTS_REVERIFIED_'
    'STAGE7C_CELL7C0_EMBEDDING_EXACT_FAISS_INDEXFLATIP_AND_COMMON_SEMANTIC_TOP20_'
    'RETRIEVAL_EXECUTION_ONLY_EXPLICITLY_AUTHORIZED_CHECKSUM_PROTECTED_NO_CELL7A3_'
    'SCORES_QUALITY_RERANKING_RRF_TOP5_PROMPTS_LLM_ANSWER_KEY_OUTCOME_INSPECTION_'
    'ADJUDICATION_OR_RAG_METRICS'
)

EXPECTED_CELL_7B5 = OrderedDict([
    ('authorization', {
        'filename': 'cell_7b5_stage7c_cell7c0_execution_authorization_v1.json',
        'sha256': '89b8848b05011b30a0b8ccabad7db35b37b671142882be8a64b1e65ec71d7a33',
    }),
    ('input_inventory', {
        'filename': 'cell_7b5_authorized_execution_input_inventory_v1.csv',
        'sha256': 'b3bec9bd8bb4a547766cfef347915d4f358fbc8ce79c0cfcbaf3da2d94714f14',
    }),
    ('qc', {
        'filename': 'cell_7b5_execution_authorization_qc_v1.json',
        'sha256': '331eb405b5983ee93a9c1e645ea29376f7493b4732b1ef1fb6226633b9baaaa9',
    }),
    ('manifest', {
        'filename': 'cell_7b5_execution_authorization_manifest_v1.json',
        'sha256': '9c499430815f4a74e364301c0e579d5277d9b62bee1a6b6e76086d59d3366fe9',
    }),
])

EXPECTED_AUTHORIZATION_DECISION = (
    'AUTHORIZE_STAGE7C_CELL7C0_EMBEDDING_AND_SEMANTIC_RETRIEVAL_EXECUTION_ONLY'
)
EXPECTED_MODEL_ID = 'NeuML/pubmedbert-base-embeddings'
EXPECTED_MODEL_REVISION = 'b79526d6ef3645e0df4530322e266f24c829f5ef'
EXPECTED_CORPUS_ROWS = 100_920
EXPECTED_QUESTION_ROWS = 80
EXPECTED_DIMENSION = 768
EXPECTED_TOP_K = 20
EXPECTED_CANDIDATE_ROWS = EXPECTED_QUESTION_ROWS * EXPECTED_TOP_K
EXPECTED_CORPUS_SHA256 = '2fead04f6c0814bb87207c9c36db7475370ae626f6a326c89ce672f402339399'
EXPECTED_QUESTIONS_SHA256 = 'c76e81952fcc6a698866b64da7b7daabeb281b7b9d10e17873596095b69d95df'
EXPECTED_EMBEDDING_CONFIG_SHA256 = 'ab209e48b025652e5ddbb79891225334ffc51de62f157b430c21aef30865b807'
EXPECTED_RUNTIME_CONFIG_SHA256 = '6003c85ef151ae1d7dca530462fa1dbcf6983be4b7e92744b8e65c8d8b42b1d3'
EXPECTED_REQUIREMENTS_SHA256 = '1a894f7ba976d00563325cc3324ff91708b5699e1c618e3674502fe586828a91'
EXPECTED_CELL_7B4_MANIFEST_SHA256 = '18a5d6cb3cadca0eab950839a19022686fc6bad2c398ed87f2a48c66eff462fe'

AUTH_DIR = ROOT / 'configs' / 'stage7_rag' / 'cell_7b5_execution_authorization_v1'
CELL_7B5_PATHS = OrderedDict([
    (key, AUTH_DIR / spec['filename'])
    if key != 'qc'
    else (
        key,
        ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
        / 'cell_7b5_execution_authorization_v1' / spec['filename'],
    )
    for key, spec in EXPECTED_CELL_7B5.items()
])

DATA_DIR = ROOT / 'outputs' / 'rag_execution' / 'stage7_rag' / PACKAGE_NAME
QC_DIR = ROOT / 'outputs' / 'quality_checks' / 'stage7_rag' / PACKAGE_NAME
CONFIG_DIR = ROOT / 'configs' / 'stage7_rag' / PACKAGE_NAME

OUTPUTS = OrderedDict([
    ('model_artifact_inventory', DATA_DIR / 'cell_7c0_embedding_model_artifact_inventory_v1.json'),
    ('corpus_embeddings', DATA_DIR / 'cell_7c0_semantic_corpus_embeddings_float32_l2_v1.npy'),
    ('corpus_embedding_identity', DATA_DIR / 'cell_7c0_semantic_corpus_embedding_identity_v1.parquet'),
    ('question_embeddings', DATA_DIR / 'cell_7c0_primary_question_embeddings_float32_l2_v1.npy'),
    ('question_embedding_identity', DATA_DIR / 'cell_7c0_primary_question_embedding_identity_v1.csv'),
    ('faiss_index', DATA_DIR / 'cell_7c0_semantic_corpus_indexflatip_v1.faiss'),
    ('semantic_top20_pool', DATA_DIR / 'cell_7c0_common_semantic_top20_candidate_pool_v1.parquet'),
    ('execution_report', DATA_DIR / 'cell_7c0_embedding_and_semantic_retrieval_report_v1.json'),
    ('qc', QC_DIR / 'cell_7c0_embedding_and_semantic_retrieval_qc_v1.json'),
    ('manifest', CONFIG_DIR / 'cell_7c0_embedding_and_semantic_retrieval_manifest_v1.json'),
])

existing_outputs = [str(path) for path in OUTPUTS.values() if path.exists()]
existing_sidecars = [str(Path(str(path) + '.sha256')) for path in OUTPUTS.values() if Path(str(path) + '.sha256').exists()]
if existing_outputs or existing_sidecars:
    raise FileExistsError(
        'Fail-closed overwrite protection is active. One or more Cell 7C0 outputs already exist:\n- '
        + '\n- '.join(existing_outputs + existing_sidecars)
    )

for directory in (DATA_DIR, QC_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

STAGING_DIR = Path(tempfile.mkdtemp(prefix='ges_rag_cell_7c0_'))
STAGED = OrderedDict((key, STAGING_DIR / path.name) for key, path in OUTPUTS.items())
MODEL_SNAPSHOT_DIR = STAGING_DIR / 'model_snapshot'

# Non-frozen recovery checkpoints. These exist only to protect long GPU work
# from Colab runtime resets. They are never included in the final Cell 7C0 manifest.
CHECKPOINT_DIR = (
    ROOT / 'outputs' / 'execution_checkpoints' / 'stage7_rag'
    / 'cell_7c0_embedding_and_semantic_retrieval_v1'
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS = OrderedDict([
    ('corpus_embeddings', CHECKPOINT_DIR / 'checkpoint_corpus_embeddings_float32_l2.npy'),
    ('question_embeddings', CHECKPOINT_DIR / 'checkpoint_question_embeddings_float32_l2.npy'),
    ('embedding_meta', CHECKPOINT_DIR / 'checkpoint_embedding_metadata.json'),
    ('semantic_top20_pool', CHECKPOINT_DIR / 'checkpoint_semantic_top20_candidate_pool.parquet'),
    ('retrieval_meta', CHECKPOINT_DIR / 'checkpoint_retrieval_metadata.json'),
])

print(f'Local staging directory : {STAGING_DIR}')
print(f'Recovery checkpoint dir : {CHECKPOINT_DIR}')
print(f'Data directory          : {DATA_DIR}')
print(f'QC directory            : {QC_DIR}')
print(f'Config directory        : {CONFIG_DIR}')

Local staging directory : /tmp/ges_rag_cell_7c0_ixbnu8wu
Recovery checkpoint dir : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/execution_checkpoints/stage7_rag/cell_7c0_embedding_and_semantic_retrieval_v1
Data directory          : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/rag_execution/stage7_rag/cell_7c0_embedding_and_semantic_retrieval_v1
QC directory            : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c0_embedding_and_semantic_retrieval_v1
Config directory        : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c0_embedding_and_semantic_retrieval_v1


## 2. Checksum, stable-write, schema, and atomic-commit helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode('utf-8', errors='strict')).hexdigest()


def sidecar_path(path: Path) -> Path:
    return Path(str(path) + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    match = re.search(r'\b[0-9a-fA-F]{64}\b', text)
    if match is None:
        raise ValueError(f'No valid SHA-256 value found in sidecar: {path}')
    return match.group(0).lower()


def sidecar_is_valid(path: Path) -> bool:
    sc = sidecar_path(path)
    return path.exists() and sc.exists() and read_sidecar_hash(sc) == sha256_file(path)


def atomic_write_bytes(path: Path, payload: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with temporary.open('wb') as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)


def to_json_native(value: Any) -> Any:
    """Recursively convert NumPy/Pandas values to strict JSON-native Python values."""
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, np.generic):
        return to_json_native(value.item())
    if isinstance(value, np.ndarray):
        return [to_json_native(item) for item in value.tolist()]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): to_json_native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_json_native(item) for item in value]
    if hasattr(value, 'item'):
        try:
            return to_json_native(value.item())
        except (TypeError, ValueError):
            pass
    raise TypeError(
        f'Object of type {value.__class__.__name__} is not JSON serializable '
        'after JSON-native normalization.'
    )


def stable_write_json(path: Path, payload: Any) -> str:
    payload = to_json_native(payload)
    data = (
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + '\n'
    ).encode('utf-8')
    atomic_write_bytes(path, data)
    return sha256_file(path)


def stable_write_csv(path: Path, rows: Iterable[dict[str, Any]], fieldnames: list[str]) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with temporary.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
            extrasaction='raise',
            lineterminator='\n',
        )
        writer.writeheader()
        writer.writerows(rows)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    if temporary.exists():
        temporary.unlink()
    table = pa.Table.from_pandas(frame, preserve_index=False)
    pq.write_table(
        table,
        temporary,
        compression='zstd',
        use_dictionary=True,
        write_statistics=True,
        version='2.6',
    )
    temporary.replace(path)
    return sha256_file(path)


def write_sidecar(path: Path) -> Path:
    digest = sha256_file(path)
    sc = sidecar_path(path)
    atomic_write_bytes(sc, f'{digest}  {path.name}\n'.encode('utf-8'))
    return sc


def atomic_copy_to_target(source: Path, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(target) + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with source.open('rb') as src, temporary.open('wb') as dst:
        shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
        dst.flush()
        os.fsync(dst.fileno())
    temporary.replace(target)


def assert_exact_file(path: Path, expected_hash: str) -> None:
    if not path.exists():
        raise FileNotFoundError(path)
    observed = sha256_file(path)
    if observed != expected_hash:
        raise AssertionError(
            f'Checksum mismatch for {path.name}: expected {expected_hash}, observed {observed}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Missing or invalid SHA-256 sidecar: {path}')


def require_columns(frame: pd.DataFrame, required: list[str], label: str) -> None:
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise KeyError(
            f'{label} is missing required frozen columns: {missing}. '
            f'Observed columns: {list(frame.columns)}'
        )


def normalize_semantic_text(value: str) -> str:
    if not isinstance(value, str):
        raise TypeError('semantic text must be a Python string')
    value.encode('utf-8', errors='strict')
    text = unicodedata.normalize('NFKC', value)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'\s+', ' ', text, flags=re.UNICODE).strip()
    return text


NORMALIZATION_REFERENCE_IMPLEMENTATION = r"""
def normalize_semantic_text(value: str) -> str:
    if not isinstance(value, str):
        raise TypeError("semantic text must be a Python string")
    value.encode("utf-8", errors="strict")
    text = unicodedata.normalize("NFKC", value)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text, flags=re.UNICODE).strip()
    return text
""".strip()


def parquet_metadata(path: Path) -> dict[str, Any]:
    parquet_file = pq.ParquetFile(path)
    return {
        'rows': int(parquet_file.metadata.num_rows),
        'columns': int(parquet_file.metadata.num_columns),
        'row_groups': int(parquet_file.metadata.num_row_groups),
        'schema_names': list(parquet_file.schema_arrow.names),
    }


def package_version(name: str) -> str | None:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


def numpy_l2_norm_summary(array: np.ndarray, chunk_size: int = 8192) -> dict[str, float]:
    minimum = math.inf
    maximum = -math.inf
    maximum_abs_error = 0.0
    for start in range(0, array.shape[0], chunk_size):
        block = np.asarray(array[start:start + chunk_size], dtype=np.float32)
        norms = np.linalg.norm(block, axis=1)
        minimum = min(minimum, float(norms.min()))
        maximum = max(maximum, float(norms.max()))
        maximum_abs_error = max(maximum_abs_error, float(np.max(np.abs(norms - 1.0))))
    return {
        'minimum_l2_norm': minimum,
        'maximum_l2_norm': maximum,
        'maximum_absolute_norm_error': maximum_abs_error,
    }




def remove_checkpoint_artifacts(paths: Iterable[Path]) -> None:
    for path in paths:
        for candidate in (path, sidecar_path(path)):
            if candidate.exists():
                candidate.unlink()


def checkpoint_file_valid(path: Path) -> bool:
    return path.exists() and sidecar_is_valid(path)


def copy_with_sidecar(source: Path, target: Path) -> str:
    atomic_copy_to_target(source, target)
    write_sidecar(target)
    if not sidecar_is_valid(target):
        raise AssertionError(f'Checkpoint sidecar validation failed: {target}')
    return sha256_file(target)


def checkpoint_context_payload() -> dict[str, Any]:
    return {
        'cell_id': CELL_ID,
        'package_name': PACKAGE_NAME,
        'model_id': EXPECTED_MODEL_ID,
        'model_revision': EXPECTED_MODEL_REVISION,
        'corpus_sha256': EXPECTED_CORPUS_SHA256,
        'questions_sha256': EXPECTED_QUESTIONS_SHA256,
        'embedding_config_sha256': EXPECTED_EMBEDDING_CONFIG_SHA256,
        'runtime_config_sha256': EXPECTED_RUNTIME_CONFIG_SHA256,
        'corpus_rows': EXPECTED_CORPUS_ROWS,
        'question_rows': EXPECTED_QUESTION_ROWS,
        'dimension': EXPECTED_DIMENSION,
        'top_k': EXPECTED_TOP_K,
        'seed': FROZEN_SEED,
    }


def checkpoint_context_matches(metadata: dict[str, Any]) -> bool:
    observed = metadata.get('context', {})
    return observed == checkpoint_context_payload()


def load_json_file(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))

print('Helper functions loaded.')

Helper functions loaded.


## 3. Reverify Cell 7B5 and recover only the authorized Cell 7C0 inputs

In [4]:
# Reverify the exact four-file Cell 7B5 authorization package.
for artifact_id, spec in EXPECTED_CELL_7B5.items():
    assert_exact_file(CELL_7B5_PATHS[artifact_id], spec['sha256'])

cell_7b5_authorization = json.loads(
    CELL_7B5_PATHS['authorization'].read_text(encoding='utf-8')
)
cell_7b5_qc = json.loads(CELL_7B5_PATHS['qc'].read_text(encoding='utf-8'))
cell_7b5_manifest = json.loads(CELL_7B5_PATHS['manifest'].read_text(encoding='utf-8'))

cell_7b5_decision = str(cell_7b5_manifest.get('terminal_decision', ''))
if cell_7b5_decision != EXPECTED_CELL_7B5_DECISION:
    raise AssertionError('Cell 7B5 terminal decision does not match the frozen authorization decision.')
if cell_7b5_authorization.get('authorization_decision') != EXPECTED_AUTHORIZATION_DECISION:
    raise AssertionError('Cell 7B5 authorization decision is not exact.')
if int(cell_7b5_qc.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7B5 frozen QC contains one or more failures.')

required_output_spec = cell_7b5_authorization.get('required_cell_7c0_outputs', {})
expected_output_filenames = OrderedDict((key, path.name) for key, path in OUTPUTS.items())
if dict(required_output_spec) != dict(expected_output_filenames):
    raise AssertionError(
        'Current Cell 7C0 output filenames do not exactly match the Cell 7B5 authorization.'
    )

authorized_inputs = cell_7b5_authorization.get('authorized_inputs', {})
required_input_keys = {
    'cell_7b4_manifest',
    'embedding_retrieval_configuration',
    'runtime_determinism_configuration',
    'requirements_lock',
    'semantic_corpus',
    'primary_questions',
}
if set(authorized_inputs) != required_input_keys:
    raise AssertionError(
        f'Cell 7B5 authorized-input inventory changed: {sorted(authorized_inputs)}'
    )

AUTHORIZED_INPUT_PATHS = OrderedDict(
    (key, Path(str(authorized_inputs[key]['path'])))
    for key in sorted(required_input_keys)
)

expected_input_hashes = {
    'cell_7b4_manifest': EXPECTED_CELL_7B4_MANIFEST_SHA256,
    'embedding_retrieval_configuration': EXPECTED_EMBEDDING_CONFIG_SHA256,
    'runtime_determinism_configuration': EXPECTED_RUNTIME_CONFIG_SHA256,
    'requirements_lock': EXPECTED_REQUIREMENTS_SHA256,
    'semantic_corpus': EXPECTED_CORPUS_SHA256,
    'primary_questions': EXPECTED_QUESTIONS_SHA256,
}
for key, path in AUTHORIZED_INPUT_PATHS.items():
    authorized_hash = str(authorized_inputs[key]['sha256'])
    if authorized_hash != expected_input_hashes[key]:
        raise AssertionError(f'Authorized hash changed for {key}.')
    assert_exact_file(path, expected_input_hashes[key])

embedding_config = json.loads(
    AUTHORIZED_INPUT_PATHS['embedding_retrieval_configuration'].read_text(encoding='utf-8')
)
runtime_config = json.loads(
    AUTHORIZED_INPUT_PATHS['runtime_determinism_configuration'].read_text(encoding='utf-8')
)
requirements_lock_text = AUTHORIZED_INPUT_PATHS['requirements_lock'].read_text(encoding='utf-8')

frozen_design = cell_7b5_authorization.get('frozen_execution_design', {})
reverification_checks = OrderedDict([
    ('four_cell_7b5_artifacts_exact', len(CELL_7B5_PATHS) == 4),
    ('cell_7b5_terminal_decision_exact', cell_7b5_decision == EXPECTED_CELL_7B5_DECISION),
    ('cell_7b5_authorization_decision_exact', cell_7b5_authorization.get('authorization_decision') == EXPECTED_AUTHORIZATION_DECISION),
    ('cell_7b5_qc_zero_failures', int(cell_7b5_qc.get('failed_checks', -1)) == 0),
    ('authorized_cell_exact_7c0', cell_7b5_authorization.get('authorized_cell', {}).get('cell_id') == '7C0'),
    ('authorized_stage_exact_7c', cell_7b5_authorization.get('authorized_cell', {}).get('stage') == '7C'),
    ('authorized_once_true', cell_7b5_authorization.get('authorized_cell', {}).get('authorized_once') is True),
    ('overwrite_existing_outputs_false', cell_7b5_authorization.get('authorized_cell', {}).get('overwrite_existing_outputs') is False),
    ('six_authorized_input_hashes_exact', all(sha256_file(path) == expected_input_hashes[key] for key, path in AUTHORIZED_INPUT_PATHS.items())),
    ('six_authorized_input_sidecars_valid', all(sidecar_is_valid(path) for path in AUTHORIZED_INPUT_PATHS.values())),
    ('embedding_model_exact', frozen_design.get('embedding_model') == EXPECTED_MODEL_ID),
    ('embedding_revision_exact', frozen_design.get('embedding_revision') == EXPECTED_MODEL_REVISION),
    ('embedding_dimension_768', int(frozen_design.get('embedding_dimension', -1)) == EXPECTED_DIMENSION),
    ('embedding_dtype_float32', frozen_design.get('output_dtype') == 'float32'),
    ('embedding_l2_true', frozen_design.get('l2_normalization') is True),
    ('faiss_indexflatip_exact', frozen_design.get('faiss_index') == 'IndexFlatIP'),
    ('approximate_search_false', frozen_design.get('approximate_search') is False),
    ('top20_exact', int(frozen_design.get('semantic_candidate_pool_k', -1)) == EXPECTED_TOP_K),
    ('common_pool_true', frozen_design.get('common_pool_for_all_conditions') is True),
    ('hard_exclusion_false', frozen_design.get('hard_evidence_exclusion') is False),
    ('global_seed_exact', int(frozen_design.get('global_seed', -1)) == FROZEN_SEED),
    ('normalization_reference_hash_exact', embedding_config['text_normalization']['reference_implementation_sha256'] == sha256_text(NORMALIZATION_REFERENCE_IMPLEMENTATION)),
    ('requirements_lock_exact_text', requirements_lock_text == (
        '# Cell 7B4 frozen execution requirements\n'
        '# Install these exact future-execution packages before any Cell 7C execution.\n'
        'sentence-transformers==5.6.0\n'
        'transformers==5.14.1\n'
        'faiss-cpu==1.14.3\n'
        'openai==2.45.0\n'
    )),
])

failed_reverification = [name for name, passed in reverification_checks.items() if not bool(passed)]
if failed_reverification:
    raise RuntimeError('Cell 7B5 reverification failed:\n- ' + '\n- '.join(failed_reverification))

immutable_hashes_before = OrderedDict(
    (key, sha256_file(path)) for key, path in AUTHORIZED_INPUT_PATHS.items()
)

print('Cell 7B5 authorization package reverified: 4/4 exact hashes + sidecars')
print('Authorized score-blind inputs reverified : 6/6 exact hashes + sidecars')
print('Cell 7A3 score table opened               : NO')
print('Answer-key artifact opened                : NO')

Cell 7B5 authorization package reverified: 4/4 exact hashes + sidecars
Authorized score-blind inputs reverified : 6/6 exact hashes + sidecars
Cell 7A3 score table opened               : NO
Answer-key artifact opened                : NO


## 4. Load, structurally validate, schema-adapt, sort, and normalize only the authorized score-blind corpus and questions

**Correction:** the checksum-frozen Cell 7B3 semantic corpus uses the source columns `evidence_packet_id` and `semantic_evidence_text`. This cell maps those exact frozen source names to the canonical internal roles `packet_id` and `semantic_text` without changing the upstream artifact.


In [5]:
load_started = time.perf_counter()

semantic_corpus_path = AUTHORIZED_INPUT_PATHS['semantic_corpus']
primary_questions_path = AUTHORIZED_INPUT_PATHS['primary_questions']

corpus_meta = parquet_metadata(semantic_corpus_path)
semantic_corpus = pd.read_parquet(semantic_corpus_path)
primary_questions = pd.read_csv(
    primary_questions_path,
    dtype=str,
    keep_default_na=False,
    encoding='utf-8',
)

semantic_corpus_source_columns = list(semantic_corpus.columns)
primary_question_source_columns = list(primary_questions.columns)

EXPECTED_SEMANTIC_CORPUS_SOURCE_COLUMNS = [
    'evidence_packet_id',
    'rcv_accession',
    'target_gene',
    'semantic_evidence_text',
]
if set(semantic_corpus_source_columns) != set(EXPECTED_SEMANTIC_CORPUS_SOURCE_COLUMNS):
    raise KeyError(
        'Unexpected checksum-frozen semantic-corpus source schema. '
        f'Expected exactly {EXPECTED_SEMANTIC_CORPUS_SOURCE_COLUMNS}; '
        f'observed {semantic_corpus_source_columns}.'
    )

SEMANTIC_CORPUS_SOURCE_TO_CANONICAL = OrderedDict([
    ('evidence_packet_id', 'packet_id'),
    ('semantic_evidence_text', 'semantic_text'),
])
semantic_corpus = semantic_corpus.rename(columns=SEMANTIC_CORPUS_SOURCE_TO_CANONICAL)

CORPUS_REQUIRED_COLUMNS = ['packet_id', 'rcv_accession', 'semantic_text']
QUESTION_REQUIRED_COLUMNS = ['question_id', 'question_text']
require_columns(semantic_corpus, CORPUS_REQUIRED_COLUMNS, 'semantic corpus after canonical-role mapping')
require_columns(primary_questions, QUESTION_REQUIRED_COLUMNS, 'primary question set')

prohibited_patterns = [
    r'full_ges',
    r'no_star_ges',
    r'combined_metadata',
    r'p_stable',
    r'instability_risk',
    r'quality_rank',
    r'semantic_rank',
    r'rrf',
    r'future_instability',
    r'answer_key',
    r'reference_answer',
    r'recommendation',
]
prohibited_corpus_columns = [
    column for column in semantic_corpus_source_columns
    if any(re.search(pattern, column, flags=re.IGNORECASE) for pattern in prohibited_patterns)
]
prohibited_question_columns = [
    column for column in primary_question_source_columns
    if any(re.search(pattern, column, flags=re.IGNORECASE) for pattern in prohibited_patterns)
]
if prohibited_corpus_columns or prohibited_question_columns:
    raise AssertionError(
        'Prohibited score/outcome fields detected in authorized score-blind inputs: '
        f'corpus={prohibited_corpus_columns}, questions={prohibited_question_columns}'
    )

semantic_corpus = semantic_corpus.copy()
primary_questions = primary_questions.copy()
semantic_corpus['_source_row_index'] = np.arange(len(semantic_corpus), dtype=np.int64)
primary_questions['_source_row_index'] = np.arange(len(primary_questions), dtype=np.int64)

for column in CORPUS_REQUIRED_COLUMNS:
    if semantic_corpus[column].isna().any():
        raise AssertionError(f'Null values detected in semantic-corpus column {column}.')
    semantic_corpus[column] = semantic_corpus[column].astype(str)
for column in QUESTION_REQUIRED_COLUMNS:
    if primary_questions[column].isna().any():
        raise AssertionError(f'Null values detected in primary-question column {column}.')
    primary_questions[column] = primary_questions[column].astype(str)

semantic_corpus = semantic_corpus.sort_values(
    ['packet_id', 'rcv_accession'],
    kind='mergesort',
).reset_index(drop=True)
primary_questions = primary_questions.sort_values(
    ['question_id'],
    kind='mergesort',
).reset_index(drop=True)

semantic_corpus['normalized_semantic_text'] = [
    normalize_semantic_text(value) for value in semantic_corpus['semantic_text'].tolist()
]
primary_questions['normalized_question_text'] = [
    normalize_semantic_text(value) for value in primary_questions['question_text'].tolist()
]

corpus_normalization_changes = int(
    (semantic_corpus['semantic_text'] != semantic_corpus['normalized_semantic_text']).sum()
)
question_normalization_changes = int(
    (primary_questions['question_text'] != primary_questions['normalized_question_text']).sum()
)

input_structure_checks = OrderedDict([
    ('semantic_corpus_source_schema_exact', set(semantic_corpus_source_columns) == set(EXPECTED_SEMANTIC_CORPUS_SOURCE_COLUMNS)),
    ('semantic_corpus_source_packet_role_exact', SEMANTIC_CORPUS_SOURCE_TO_CANONICAL['evidence_packet_id'] == 'packet_id'),
    ('semantic_corpus_source_text_role_exact', SEMANTIC_CORPUS_SOURCE_TO_CANONICAL['semantic_evidence_text'] == 'semantic_text'),
    ('semantic_corpus_metadata_rows_100920', corpus_meta['rows'] == EXPECTED_CORPUS_ROWS),
    ('semantic_corpus_loaded_rows_100920', len(semantic_corpus) == EXPECTED_CORPUS_ROWS),
    ('primary_questions_rows_80', len(primary_questions) == EXPECTED_QUESTION_ROWS),
    ('packet_ids_complete', semantic_corpus['packet_id'].str.len().gt(0).all()),
    ('packet_ids_unique', semantic_corpus['packet_id'].nunique(dropna=False) == EXPECTED_CORPUS_ROWS),
    ('rcv_accessions_complete', semantic_corpus['rcv_accession'].str.len().gt(0).all()),
    ('rcv_accessions_unique', semantic_corpus['rcv_accession'].nunique(dropna=False) == EXPECTED_CORPUS_ROWS),
    ('semantic_text_complete', semantic_corpus['normalized_semantic_text'].str.len().gt(0).all()),
    ('question_ids_complete', primary_questions['question_id'].str.len().gt(0).all()),
    ('question_ids_unique', primary_questions['question_id'].nunique(dropna=False) == EXPECTED_QUESTION_ROWS),
    ('question_text_complete', primary_questions['normalized_question_text'].str.len().gt(0).all()),
    ('corpus_sorted_packet_then_rcv', semantic_corpus[['packet_id', 'rcv_accession']].equals(
        semantic_corpus[['packet_id', 'rcv_accession']].sort_values(['packet_id', 'rcv_accession'], kind='mergesort').reset_index(drop=True)
    )),
    ('questions_sorted_question_id', primary_questions[['question_id']].equals(
        primary_questions[['question_id']].sort_values(['question_id'], kind='mergesort').reset_index(drop=True)
    )),
    ('corpus_no_prohibited_columns', len(prohibited_corpus_columns) == 0),
    ('questions_no_prohibited_columns', len(prohibited_question_columns) == 0),
    ('answer_key_not_loaded', True),
    ('cell_7a3_scores_not_loaded', True),
])
failed_input_structure = [name for name, passed in input_structure_checks.items() if not bool(passed)]
if failed_input_structure:
    raise RuntimeError('Authorized-input structural validation failed:\n- ' + '\n- '.join(failed_input_structure))

load_elapsed_seconds = time.perf_counter() - load_started
print(f'Semantic corpus source schema         : {semantic_corpus_source_columns}')
print('Canonical role mapping                : evidence_packet_id→packet_id; semantic_evidence_text→semantic_text')
print(f'Semantic corpus loaded and normalized : {len(semantic_corpus):,} rows')
print(f'Primary questions loaded and normalized: {len(primary_questions):,} rows')
print(f'Corpus normalization changes          : {corpus_normalization_changes:,}')
print(f'Question normalization changes        : {question_normalization_changes:,}')
print('Frozen upstream artifact modified     : NO')
print('Question contents displayed           : NO')
print('Semantic retrieval results displayed  : NO')


Semantic corpus source schema         : ['evidence_packet_id', 'rcv_accession', 'target_gene', 'semantic_evidence_text']
Canonical role mapping                : evidence_packet_id→packet_id; semantic_evidence_text→semantic_text
Semantic corpus loaded and normalized : 100,920 rows
Primary questions loaded and normalized: 80 rows
Corpus normalization changes          : 100,920
Question normalization changes        : 0
Frozen upstream artifact modified     : NO
Question contents displayed           : NO
Semantic retrieval results displayed  : NO


In [6]:
runtime_started = time.perf_counter()

observed_packages = OrderedDict([
    ('sentence-transformers', package_version('sentence-transformers')),
    ('transformers', package_version('transformers')),
    ('faiss-cpu', package_version('faiss-cpu')),
    ('openai', package_version('openai')),
    ('numpy', package_version('numpy')),
    ('pandas', package_version('pandas')),
    ('pyarrow', package_version('pyarrow')),
    ('torch', package_version('torch')),
    ('huggingface-hub', package_version('huggingface-hub')),
])
expected_package_versions = {
    'sentence-transformers': '5.6.0',
    'transformers': '5.14.1',
    'faiss-cpu': '1.14.3',
    'openai': '2.45.0',
}
package_mismatches = {
    name: {'expected': expected, 'observed': observed_packages.get(name)}
    for name, expected in expected_package_versions.items()
    if observed_packages.get(name) != expected
}
if package_mismatches:
    raise RuntimeError(f'Frozen package-version mismatch: {package_mismatches}')

random.seed(FROZEN_SEED)
np.random.seed(FROZEN_SEED)
torch.manual_seed(FROZEN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(FROZEN_SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Cell 7C0 contains 100,920 PubMedBERT forward passes and is not permitted
# to silently fall back to CPU. A CUDA GPU is therefore an execution requirement
# for this notebook implementation.
if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA GPU is not available. Stop here, select Runtime > Change runtime type '
        '> T4 GPU (or another GPU), reconnect, and run the notebook from Cell 1. '
        'CPU execution would take many hours and is intentionally prohibited.'
    )

DEVICE = 'cuda'
GPU_NAME = torch.cuda.get_device_name(0)
CUDA_RUNTIME_VERSION = torch.version.cuda
GPU_CAPABILITY = torch.cuda.get_device_capability(0)

api = HfApi()
model_info = api.model_info(EXPECTED_MODEL_ID, revision=EXPECTED_MODEL_REVISION)
resolved_revision = str(model_info.sha)
if resolved_revision != EXPECTED_MODEL_REVISION:
    raise AssertionError(
        f'Hugging Face revision mismatch: expected {EXPECTED_MODEL_REVISION}, observed {resolved_revision}'
    )

snapshot_path = Path(snapshot_download(
    repo_id=EXPECTED_MODEL_ID,
    revision=EXPECTED_MODEL_REVISION,
    local_dir=MODEL_SNAPSHOT_DIR,
))

model_files = []
for path in sorted(snapshot_path.rglob('*')):
    if not path.is_file():
        continue
    relative = path.relative_to(snapshot_path).as_posix()
    if relative.startswith('.cache/'):
        continue
    model_files.append({
        'relative_path': relative,
        'bytes': int(path.stat().st_size),
        'sha256': sha256_file(path),
    })
if not model_files:
    raise AssertionError('The exact model snapshot inventory is empty.')

model = SentenceTransformer(
    str(snapshot_path),
    device=DEVICE,
    trust_remote_code=False,
)
model.to(torch.device(DEVICE))
model.eval()
model.float()

model_parameter_device = str(next(model.parameters()).device)
if not model_parameter_device.startswith('cuda'):
    raise RuntimeError(
        f'The embedding model is not resident on CUDA: {model_parameter_device}'
    )

if hasattr(model, 'get_embedding_dimension'):
    observed_dimension = int(model.get_embedding_dimension())
else:
    observed_dimension = int(model.get_sentence_embedding_dimension())
observed_max_seq_length = int(model.max_seq_length)
if observed_dimension != EXPECTED_DIMENSION:
    raise AssertionError(
        f'Unexpected embedding dimension: expected {EXPECTED_DIMENSION}, observed {observed_dimension}'
    )
if observed_max_seq_length != 512:
    raise AssertionError(
        f'Unexpected model max sequence length: expected 512, observed {observed_max_seq_length}'
    )

pooling_modules = [
    module for module in model._modules.values()
    if module.__class__.__name__.lower() == 'pooling'
]
if len(pooling_modules) != 1:
    raise AssertionError(f'Expected exactly one SentenceTransformers Pooling module; found {len(pooling_modules)}.')
pooling_module = pooling_modules[0]
pooling_config = pooling_module.get_config_dict()

# Sentence-Transformers 5.x may expose Pooling configuration in either:
#   1) the newer compact form: {'pooling_mode': 'mean', ...}, or
#   2) the legacy boolean-flag form:
#      pooling_mode_mean_tokens=True, pooling_mode_cls_token=False, etc.
# Both representations describe the same frozen mean-token pooling behavior.
compact_pooling_mode = str(pooling_config.get('pooling_mode', '')).strip().lower()
legacy_pooling_keys_present = any(
    key in pooling_config
    for key in [
        'pooling_mode_mean_tokens',
        'pooling_mode_cls_token',
        'pooling_mode_max_tokens',
        'pooling_mode_mean_sqrt_len_tokens',
        'pooling_mode_weightedmean_tokens',
        'pooling_mode_lasttoken',
    ]
)

if compact_pooling_mode:
    mean_pooling_exact = compact_pooling_mode == 'mean'
    pooling_validation_representation = 'compact_pooling_mode'
elif legacy_pooling_keys_present:
    mean_pooling_exact = (
        bool(pooling_config.get('pooling_mode_mean_tokens'))
        and not bool(pooling_config.get('pooling_mode_cls_token'))
        and not bool(pooling_config.get('pooling_mode_max_tokens'))
        and not bool(pooling_config.get('pooling_mode_mean_sqrt_len_tokens'))
        and not bool(pooling_config.get('pooling_mode_weightedmean_tokens'))
        and not bool(pooling_config.get('pooling_mode_lasttoken'))
    )
    pooling_validation_representation = 'legacy_boolean_flags'
else:
    raise AssertionError(
        'Unable to determine the frozen pooling mode from the SentenceTransformers '
        f'configuration: {pooling_config}'
    )

if not mean_pooling_exact:
    raise AssertionError(
        'Frozen model is not configured for exact mean-token pooling: '
        f'{pooling_config}'
    )

# The frozen text-normalization policy preserves case before tokenization.
# The pinned embedding model is based on an uncased PubMedBERT tokenizer, so
# model-native lowercasing is an intrinsic property of the exact frozen model
# revision and must be recorded rather than rejected.
external_case_probe = 'BRCA1 EGFR ClinVar Pathogenic'
external_case_normalized = normalize_semantic_text(external_case_probe)
external_text_normalization_preserves_case = (
    external_case_normalized == external_case_probe
)
if not external_text_normalization_preserves_case:
    raise AssertionError(
        'The frozen external text-normalization implementation unexpectedly changed case.'
    )

tokenizer_do_lower_case = bool(
    getattr(model.tokenizer, 'do_lower_case', False)
)
upper_probe_ids = model.tokenizer(
    'Cancer Variant Evidence',
    add_special_tokens=False,
)['input_ids']
lower_probe_ids = model.tokenizer(
    'cancer variant evidence',
    add_special_tokens=False,
)['input_ids']
tokenizer_case_probe_equivalent = upper_probe_ids == lower_probe_ids

if tokenizer_do_lower_case and not tokenizer_case_probe_equivalent:
    raise AssertionError(
        'Tokenizer reports lowercasing but the deterministic case probe produced '
        'different token IDs.'
    )

tokenizer_case_behavior = (
    'model_native_uncased_tokenization'
    if tokenizer_do_lower_case
    else 'model_native_case_sensitive_tokenization'
)

model_inventory_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'model_id': EXPECTED_MODEL_ID,
    'requested_revision': EXPECTED_MODEL_REVISION,
    'resolved_revision': resolved_revision,
    'provider': 'huggingface_hub',
    'library': 'sentence-transformers',
    'device': DEVICE,
    'gpu_name': GPU_NAME,
    'cuda_runtime_version': CUDA_RUNTIME_VERSION,
    'gpu_compute_capability': list(GPU_CAPABILITY),
    'model_parameter_device': model_parameter_device,
    'embedding_dimension': observed_dimension,
    'max_sequence_length_tokens': observed_max_seq_length,
    'pooling_configuration': pooling_config,
    'pooling_validation_representation': pooling_validation_representation,
    'mean_token_pooling_verified': bool(mean_pooling_exact),
    'external_text_normalization_preserves_case': bool(external_text_normalization_preserves_case),
    'tokenizer_do_lower_case': bool(tokenizer_do_lower_case),
    'tokenizer_case_probe_equivalent': bool(tokenizer_case_probe_equivalent),
    'tokenizer_case_behavior': tokenizer_case_behavior,
    'case_handling_interpretation': (
        'Case is preserved by the frozen external normalization step; '
        'the exact pinned model then applies its intrinsic tokenizer behavior.'
    ),
    'trust_remote_code': False,
    'snapshot_file_count': len(model_files),
    'snapshot_total_bytes': int(sum(row['bytes'] for row in model_files)),
    'snapshot_files': model_files,
}
stable_write_json(STAGED['model_artifact_inventory'], model_inventory_payload)

model_load_elapsed_seconds = time.perf_counter() - runtime_started
print(f'Embedding model loaded: {EXPECTED_MODEL_ID}')
print(f'Exact revision verified: {resolved_revision}')
print(f'Device                 : {DEVICE}')
print(f'GPU                    : {GPU_NAME}')
print(f'Model parameter device : {model_parameter_device}')
print(f'CUDA runtime           : {CUDA_RUNTIME_VERSION}')
print(f'Model snapshot files   : {len(model_files):,}')
print(f'Embedding dimension    : {observed_dimension}')
print(f'Pooling mode verified  : mean ({pooling_validation_representation})')
print(f'External case preserved: {external_text_normalization_preserves_case}')
print(f'Tokenizer case behavior: {tokenizer_case_behavior}')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded: NeuML/pubmedbert-base-embeddings
Exact revision verified: b79526d6ef3645e0df4530322e266f24c829f5ef
Device                 : cuda
GPU                    : NVIDIA A100-SXM4-80GB
Model parameter device : cuda:0
CUDA runtime           : 12.8
Model snapshot files   : 15
Embedding dimension    : 768
Pooling mode verified  : mean (compact_pooling_mode)
External case preserved: True
Tokenizer case behavior: model_native_uncased_tokenization


## 6. Generate and freeze float32 L2-normalized corpus and question embeddings

In [7]:
embedding_stage_started = time.perf_counter()

if DEVICE != 'cuda' or not torch.cuda.is_available():
    raise RuntimeError('Cell 7C0 embedding execution requires an active CUDA GPU.')
if not str(next(model.parameters()).device).startswith('cuda'):
    raise RuntimeError(
        f'Model moved away from CUDA before embedding execution: {next(model.parameters()).device}'
    )

print(f'Beginning embedding stage on GPU : {torch.cuda.get_device_name(0)}')
print(f'Model device                     : {next(model.parameters()).device}')

BATCH_SIZE = int(embedding_config['embedding']['batch_size'])
if BATCH_SIZE != 64:
    raise AssertionError(f'Frozen embedding batch size changed: {BATCH_SIZE}')

corpus_texts = semantic_corpus['normalized_semantic_text'].tolist()
question_texts = primary_questions['normalized_question_text'].tolist()

embedding_checkpoint_reused = False
embedding_generation_seconds = None

candidate_checkpoint = (
    checkpoint_file_valid(CHECKPOINTS['corpus_embeddings'])
    and checkpoint_file_valid(CHECKPOINTS['question_embeddings'])
    and checkpoint_file_valid(CHECKPOINTS['embedding_meta'])
)

if candidate_checkpoint:
    try:
        embedding_meta = load_json_file(CHECKPOINTS['embedding_meta'])
        if not checkpoint_context_matches(embedding_meta):
            raise AssertionError('Embedding checkpoint context mismatch.')

        cp_corpus = np.load(CHECKPOINTS['corpus_embeddings'], mmap_mode='r')
        cp_questions = np.load(CHECKPOINTS['question_embeddings'], mmap_mode='r')

        if cp_corpus.shape != (EXPECTED_CORPUS_ROWS, EXPECTED_DIMENSION):
            raise AssertionError(f'Checkpoint corpus shape mismatch: {cp_corpus.shape}')
        if cp_questions.shape != (EXPECTED_QUESTION_ROWS, EXPECTED_DIMENSION):
            raise AssertionError(f'Checkpoint question shape mismatch: {cp_questions.shape}')
        if cp_corpus.dtype != np.dtype('float32') or cp_questions.dtype != np.dtype('float32'):
            raise AssertionError('Checkpoint embedding dtype mismatch.')
        if sha256_file(CHECKPOINTS['corpus_embeddings']) != embedding_meta['corpus_embeddings_sha256']:
            raise AssertionError('Checkpoint corpus hash mismatch.')
        if sha256_file(CHECKPOINTS['question_embeddings']) != embedding_meta['question_embeddings_sha256']:
            raise AssertionError('Checkpoint question hash mismatch.')

        atomic_copy_to_target(CHECKPOINTS['corpus_embeddings'], STAGED['corpus_embeddings'])
        atomic_copy_to_target(CHECKPOINTS['question_embeddings'], STAGED['question_embeddings'])
        embedding_generation_seconds = float(embedding_meta['embedding_generation_seconds'])
        embedding_checkpoint_reused = True
        print('Validated embedding checkpoint   : REUSED')
    except Exception as exc:
        print(f'Embedding checkpoint rejected    : {exc}')
        remove_checkpoint_artifacts([
            CHECKPOINTS['corpus_embeddings'],
            CHECKPOINTS['question_embeddings'],
            CHECKPOINTS['embedding_meta'],
        ])

if not embedding_checkpoint_reused:
    print('Validated embedding checkpoint   : NOT AVAILABLE — generating')
    generation_started = time.perf_counter()

    corpus_memmap = np.lib.format.open_memmap(
        STAGED['corpus_embeddings'],
        mode='w+',
        dtype=np.float32,
        shape=(EXPECTED_CORPUS_ROWS, EXPECTED_DIMENSION),
    )

    with torch.inference_mode():
        for start in tqdm(
            range(0, EXPECTED_CORPUS_ROWS, BATCH_SIZE),
            desc='Encoding frozen semantic corpus',
        ):
            stop = min(start + BATCH_SIZE, EXPECTED_CORPUS_ROWS)
            block = model.encode(
                corpus_texts[start:stop],
                batch_size=BATCH_SIZE,
                show_progress_bar=False,
                convert_to_numpy=True,
                convert_to_tensor=False,
                normalize_embeddings=True,
                precision='float32',
            )
            block = np.ascontiguousarray(block, dtype=np.float32)
            if block.shape != (stop - start, EXPECTED_DIMENSION):
                raise AssertionError(f'Unexpected corpus embedding block shape: {block.shape}')
            if not np.isfinite(block).all():
                raise AssertionError('Non-finite corpus embedding value detected.')
            corpus_memmap[start:stop] = block

    corpus_memmap.flush()
    del corpus_memmap
    gc.collect()
    torch.cuda.empty_cache()

    with torch.inference_mode():
        question_embeddings = model.encode(
            question_texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            convert_to_tensor=False,
            normalize_embeddings=True,
            precision='float32',
        )

    question_embeddings = np.ascontiguousarray(question_embeddings, dtype=np.float32)
    if question_embeddings.shape != (EXPECTED_QUESTION_ROWS, EXPECTED_DIMENSION):
        raise AssertionError(f'Unexpected question embedding shape: {question_embeddings.shape}')
    if not np.isfinite(question_embeddings).all():
        raise AssertionError('Non-finite question embedding value detected.')

    np.save(STAGED['question_embeddings'], question_embeddings, allow_pickle=False)
    embedding_generation_seconds = time.perf_counter() - generation_started

corpus_embeddings = np.load(STAGED['corpus_embeddings'], mmap_mode='r')
question_embeddings_readback = np.load(STAGED['question_embeddings'], mmap_mode='r')

corpus_norm_summary = numpy_l2_norm_summary(corpus_embeddings)
question_norm_summary = numpy_l2_norm_summary(question_embeddings_readback)

embedding_checks = OrderedDict([
    ('corpus_embedding_shape_exact', corpus_embeddings.shape == (EXPECTED_CORPUS_ROWS, EXPECTED_DIMENSION)),
    ('question_embedding_shape_exact', question_embeddings_readback.shape == (EXPECTED_QUESTION_ROWS, EXPECTED_DIMENSION)),
    ('corpus_embedding_dtype_float32', corpus_embeddings.dtype == np.dtype('float32')),
    ('question_embedding_dtype_float32', question_embeddings_readback.dtype == np.dtype('float32')),
    ('corpus_embeddings_all_finite', np.isfinite(np.asarray(corpus_embeddings[::4096])).all()),
    ('question_embeddings_all_finite', np.isfinite(np.asarray(question_embeddings_readback)).all()),
    ('corpus_l2_norm_error_within_1e_5', corpus_norm_summary['maximum_absolute_norm_error'] <= 1e-5),
    ('question_l2_norm_error_within_1e_5', question_norm_summary['maximum_absolute_norm_error'] <= 1e-5),
])
failed_embedding_checks = [name for name, passed in embedding_checks.items() if not bool(passed)]
if failed_embedding_checks:
    raise RuntimeError('Embedding QC failed:\n- ' + '\n- '.join(failed_embedding_checks))

# Save expensive work only after QC has passed.
if not embedding_checkpoint_reused:
    corpus_cp_sha = copy_with_sidecar(STAGED['corpus_embeddings'], CHECKPOINTS['corpus_embeddings'])
    question_cp_sha = copy_with_sidecar(STAGED['question_embeddings'], CHECKPOINTS['question_embeddings'])

    embedding_meta = {
        'context': checkpoint_context_payload(),
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'embedding_generation_seconds': float(embedding_generation_seconds),
        'corpus_embeddings_sha256': corpus_cp_sha,
        'question_embeddings_sha256': question_cp_sha,
        'corpus_shape': [EXPECTED_CORPUS_ROWS, EXPECTED_DIMENSION],
        'question_shape': [EXPECTED_QUESTION_ROWS, EXPECTED_DIMENSION],
        'dtype': 'float32',
        'l2_normalized': True,
    }
    stable_write_json(CHECKPOINTS['embedding_meta'], embedding_meta)
    write_sidecar(CHECKPOINTS['embedding_meta'])
    if not sidecar_is_valid(CHECKPOINTS['embedding_meta']):
        raise AssertionError('Embedding metadata checkpoint sidecar failed.')
    print('Embedding recovery checkpoint     : SAVED TO DRIVE')

embedding_elapsed_seconds = time.perf_counter() - embedding_stage_started

print(f'Corpus embeddings                 : {corpus_embeddings.shape[0]:,} × {corpus_embeddings.shape[1]} float32, L2-normalized')
print(f'Question embeddings               : {question_embeddings_readback.shape[0]:,} × {question_embeddings_readback.shape[1]} float32, L2-normalized')
print(f'Embedding checkpoint reused       : {embedding_checkpoint_reused}')
print(f'Original generation runtime       : {embedding_generation_seconds:,.2f} seconds')
print(f'Current embedding-stage runtime   : {embedding_elapsed_seconds:,.2f} seconds')


Beginning embedding stage on GPU : NVIDIA A100-SXM4-80GB
Model device                     : cuda:0
Validated embedding checkpoint   : NOT AVAILABLE — generating


Encoding frozen semantic corpus:   0%|          | 0/1577 [00:00<?, ?it/s]

Embedding recovery checkpoint     : SAVED TO DRIVE
Corpus embeddings                 : 100,920 × 768 float32, L2-normalized
Question embeddings               : 80 × 768 float32, L2-normalized
Embedding checkpoint reused       : False
Original generation runtime       : 750.28 seconds
Current embedding-stage runtime   : 754.23 seconds


## 7. Construct the exact FAISS IndexFlatIP index and materialize the deterministic common semantic top-20 pool

In [8]:
retrieval_stage_started = time.perf_counter()

index = faiss.IndexFlatIP(EXPECTED_DIMENSION)
FAISS_ADD_BATCH = 8192
for start in tqdm(
    range(0, EXPECTED_CORPUS_ROWS, FAISS_ADD_BATCH),
    desc='Adding vectors to exact FAISS index',
):
    stop = min(start + FAISS_ADD_BATCH, EXPECTED_CORPUS_ROWS)
    block = np.ascontiguousarray(corpus_embeddings[start:stop], dtype=np.float32)
    index.add(block)

if int(index.ntotal) != EXPECTED_CORPUS_ROWS:
    raise AssertionError(f'FAISS index row count mismatch: {index.ntotal}')
if int(index.d) != EXPECTED_DIMENSION:
    raise AssertionError(f'FAISS index dimension mismatch: {index.d}')

faiss.write_index(index, str(STAGED['faiss_index']))

packet_ids = semantic_corpus['packet_id'].to_numpy(dtype=str)
rcv_accessions = semantic_corpus['rcv_accession'].to_numpy(dtype=str)
question_ids = primary_questions['question_id'].to_numpy(dtype=str)

retrieval_checkpoint_reused = False
retrieval_generation_seconds = None

candidate_checkpoint = (
    checkpoint_file_valid(CHECKPOINTS['semantic_top20_pool'])
    and checkpoint_file_valid(CHECKPOINTS['retrieval_meta'])
)

if candidate_checkpoint:
    try:
        retrieval_meta = load_json_file(CHECKPOINTS['retrieval_meta'])
        if not checkpoint_context_matches(retrieval_meta):
            raise AssertionError('Retrieval checkpoint context mismatch.')
        if retrieval_meta.get('corpus_embeddings_sha256') != sha256_file(CHECKPOINTS['corpus_embeddings']):
            raise AssertionError('Retrieval checkpoint embedding provenance mismatch.')

        semantic_top20 = pd.read_parquet(CHECKPOINTS['semantic_top20_pool'])
        if sha256_file(CHECKPOINTS['semantic_top20_pool']) != retrieval_meta['semantic_top20_sha256']:
            raise AssertionError('Retrieval checkpoint hash mismatch.')

        retrieval_generation_seconds = float(retrieval_meta['retrieval_generation_seconds'])
        retrieval_checkpoint_reused = True
        print('Validated top-20 checkpoint      : REUSED')
    except Exception as exc:
        print(f'Retrieval checkpoint rejected    : {exc}')
        remove_checkpoint_artifacts([
            CHECKPOINTS['semantic_top20_pool'],
            CHECKPOINTS['retrieval_meta'],
        ])

if not retrieval_checkpoint_reused:
    print('Validated top-20 checkpoint      : NOT AVAILABLE — executing retrieval')
    generation_started = time.perf_counter()

    candidate_rows: list[dict[str, Any]] = []
    for question_row_index in tqdm(
        range(EXPECTED_QUESTION_ROWS),
        desc='Exact semantic retrieval with deterministic tie resolution',
    ):
        query = np.ascontiguousarray(
            question_embeddings_readback[question_row_index:question_row_index + 1],
            dtype=np.float32,
        )
        all_scores, all_indices = index.search(query, EXPECTED_CORPUS_ROWS)
        scores = all_scores[0]
        indices = all_indices[0]
        if len(indices) != EXPECTED_CORPUS_ROWS or np.any(indices < 0):
            raise AssertionError('Exact FAISS search did not return every corpus row.')

        candidate_packet_ids = packet_ids[indices]
        candidate_rcvs = rcv_accessions[indices]
        deterministic_order = np.lexsort((candidate_packet_ids, candidate_rcvs, -scores))
        selected_positions = deterministic_order[:EXPECTED_TOP_K]

        for rank, selected_position in enumerate(selected_positions, start=1):
            corpus_row_index = int(indices[selected_position])
            candidate_rows.append({
                'question_id': str(question_ids[question_row_index]),
                'semantic_rank': int(rank),
                'semantic_score': np.float32(scores[selected_position]),
                'corpus_row_index': corpus_row_index,
                'packet_id': str(packet_ids[corpus_row_index]),
                'rcv_accession': str(rcv_accessions[corpus_row_index]),
            })

    semantic_top20 = pd.DataFrame(candidate_rows, columns=[
        'question_id',
        'semantic_rank',
        'semantic_score',
        'corpus_row_index',
        'packet_id',
        'rcv_accession',
    ])
    retrieval_generation_seconds = time.perf_counter() - generation_started

semantic_top20['question_id'] = semantic_top20['question_id'].astype('string')
semantic_top20['semantic_rank'] = semantic_top20['semantic_rank'].astype('int16')
semantic_top20['semantic_score'] = semantic_top20['semantic_score'].astype('float32')
semantic_top20['corpus_row_index'] = semantic_top20['corpus_row_index'].astype('int64')
semantic_top20['packet_id'] = semantic_top20['packet_id'].astype('string')
semantic_top20['rcv_accession'] = semantic_top20['rcv_accession'].astype('string')

retrieval_structure_checks = OrderedDict([
    ('faiss_index_class_exact', index.__class__.__name__ == 'IndexFlatIP'),
    ('faiss_index_ntotal_100920', int(index.ntotal) == EXPECTED_CORPUS_ROWS),
    ('faiss_index_dimension_768', int(index.d) == EXPECTED_DIMENSION),
    ('candidate_pool_rows_1600', len(semantic_top20) == EXPECTED_CANDIDATE_ROWS),
    ('candidate_pool_six_columns_exact', list(semantic_top20.columns) == [
        'question_id', 'semantic_rank', 'semantic_score', 'corpus_row_index', 'packet_id', 'rcv_accession'
    ]),
    ('candidate_pool_80_questions', semantic_top20['question_id'].nunique(dropna=False) == EXPECTED_QUESTION_ROWS),
    ('twenty_candidates_per_question', semantic_top20.groupby('question_id', sort=False).size().eq(EXPECTED_TOP_K).all()),
    ('ranks_one_to_twenty_per_question', all(
        group['semantic_rank'].tolist() == list(range(1, EXPECTED_TOP_K + 1))
        for _, group in semantic_top20.groupby('question_id', sort=False)
    )),
    ('candidate_rows_unique_within_question', not semantic_top20.duplicated(['question_id', 'corpus_row_index']).any()),
    ('candidate_packet_identity_exact', all(
        packet_ids[int(row.corpus_row_index)] == str(row.packet_id)
        for row in semantic_top20.itertuples(index=False)
    )),
    ('candidate_rcv_identity_exact', all(
        rcv_accessions[int(row.corpus_row_index)] == str(row.rcv_accession)
        for row in semantic_top20.itertuples(index=False)
    )),
    ('semantic_scores_all_finite', np.isfinite(semantic_top20['semantic_score'].to_numpy(dtype=np.float32)).all()),
    ('semantic_scores_cosine_bounds', semantic_top20['semantic_score'].between(-1.00001, 1.00001).all()),
    ('semantic_scores_nonincreasing_per_question', all(
        np.all(np.diff(group['semantic_score'].to_numpy(dtype=np.float64)) <= 1e-7)
        for _, group in semantic_top20.groupby('question_id', sort=False)
    )),
    ('common_pool_materialized_once', len(semantic_top20) == EXPECTED_QUESTION_ROWS * EXPECTED_TOP_K),
    ('quality_columns_absent', not any(
        token in column.lower()
        for column in semantic_top20.columns
        for token in ['quality', 'ges', 'rrf', 'combined_metadata', 'review_star']
    )),
    ('top5_not_materialized', int(semantic_top20['semantic_rank'].max()) == EXPECTED_TOP_K),
    ('prompts_not_materialized', True),
    ('llm_not_called', True),
    ('answer_key_not_loaded', True),
    ('rag_metrics_not_calculated', True),
])
failed_retrieval_checks = [name for name, passed in retrieval_structure_checks.items() if not bool(passed)]
if failed_retrieval_checks:
    raise RuntimeError('Semantic retrieval QC failed:\n- ' + '\n- '.join(failed_retrieval_checks))

stable_write_parquet(STAGED['semantic_top20_pool'], semantic_top20)

if not retrieval_checkpoint_reused:
    top20_sha = copy_with_sidecar(STAGED['semantic_top20_pool'], CHECKPOINTS['semantic_top20_pool'])
    retrieval_meta = {
        'context': checkpoint_context_payload(),
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'retrieval_generation_seconds': float(retrieval_generation_seconds),
        'corpus_embeddings_sha256': sha256_file(CHECKPOINTS['corpus_embeddings']),
        'semantic_top20_sha256': top20_sha,
        'candidate_rows': EXPECTED_CANDIDATE_ROWS,
        'questions': EXPECTED_QUESTION_ROWS,
        'top_k': EXPECTED_TOP_K,
    }
    stable_write_json(CHECKPOINTS['retrieval_meta'], retrieval_meta)
    write_sidecar(CHECKPOINTS['retrieval_meta'])
    if not sidecar_is_valid(CHECKPOINTS['retrieval_meta']):
        raise AssertionError('Retrieval metadata checkpoint sidecar failed.')
    print('Top-20 recovery checkpoint       : SAVED TO DRIVE')

retrieval_elapsed_seconds = time.perf_counter() - retrieval_stage_started

print(f'Exact FAISS index vectors        : {index.ntotal:,}')
print(f'Common semantic candidate rows   : {len(semantic_top20):,}')
print(f'Questions × top-k                : {EXPECTED_QUESTION_ROWS} × {EXPECTED_TOP_K}')
print(f'Retrieval checkpoint reused      : {retrieval_checkpoint_reused}')
print(f'Original retrieval runtime       : {retrieval_generation_seconds:,.2f} seconds')
print(f'Current retrieval-stage runtime  : {retrieval_elapsed_seconds:,.2f} seconds')
print('Candidate content displayed      : NO')


Adding vectors to exact FAISS index:   0%|          | 0/13 [00:00<?, ?it/s]

Validated top-20 checkpoint      : NOT AVAILABLE — executing retrieval


Exact semantic retrieval with deterministic tie resolution:   0%|          | 0/80 [00:00<?, ?it/s]

Top-20 recovery checkpoint       : SAVED TO DRIVE
Exact FAISS index vectors        : 100,920
Common semantic candidate rows   : 1,600
Questions × top-k                : 80 × 20
Retrieval checkpoint reused      : False
Original retrieval runtime       : 9.13 seconds
Current retrieval-stage runtime  : 10.99 seconds
Candidate content displayed      : NO


## 8. Freeze identity maps, execution lineage, prewrite QC, and the manifest

In [9]:
# A runtime reset cannot be repaired by importing pandas alone because Section 8
# depends on objects produced by earlier cells. Run the whole notebook from the
# beginning; validated V6 Drive checkpoints make the expensive stages resumable.
_REQUIRED_SECTION8_STATE = [
    'pd', 'np', 'faiss', 'semantic_corpus', 'primary_questions',
    'corpus_embeddings', 'question_embeddings_readback', 'semantic_top20',
    'index', 'STAGED', 'AUTHORIZED_INPUT_PATHS', 'embedding_checks',
    'retrieval_structure_checks', 'reverification_checks', 'input_structure_checks',
]
_missing_section8_state = [
    name for name in _REQUIRED_SECTION8_STATE if name not in globals()
]
if _missing_section8_state:
    raise RuntimeError(
        'Colab runtime state reset before Section 8. Missing objects: '
        + ', '.join(_missing_section8_state)
        + '. Use Runtime > Run all. Valid V6 checkpoints will be reused, so '
          'the 100,920-record embeddings will not be regenerated when the '
          'checkpoint is present and valid.'
    )

# Identity maps deliberately contain identifiers and text hashes, not semantic text or answers.
corpus_identity = pd.DataFrame({
    'corpus_row_index': np.arange(EXPECTED_CORPUS_ROWS, dtype=np.int64),
    'source_row_index': semantic_corpus['_source_row_index'].to_numpy(dtype=np.int64),
    'packet_id': semantic_corpus['packet_id'].astype(str).to_numpy(),
    'rcv_accession': semantic_corpus['rcv_accession'].astype(str).to_numpy(),
    'semantic_text_sha256': [
        sha256_text(value) for value in semantic_corpus['normalized_semantic_text'].tolist()
    ],
})
stable_write_parquet(STAGED['corpus_embedding_identity'], corpus_identity)

question_identity_rows = [
    {
        'question_row_index': int(index_value),
        'source_row_index': int(row['_source_row_index']),
        'question_id': str(row['question_id']),
        'question_text_sha256': sha256_text(str(row['normalized_question_text'])),
    }
    for index_value, row in primary_questions.iterrows()
]
stable_write_csv(
    STAGED['question_embedding_identity'],
    question_identity_rows,
    ['question_row_index', 'source_row_index', 'question_id', 'question_text_sha256'],
)

# Reopen the exact persisted index and artifacts before QC freeze.
index_readback = faiss.read_index(str(STAGED['faiss_index']))
corpus_embeddings_readback = np.load(STAGED['corpus_embeddings'], mmap_mode='r')
question_embeddings_final = np.load(STAGED['question_embeddings'], mmap_mode='r')
semantic_top20_readback = pd.read_parquet(STAGED['semantic_top20_pool'])
corpus_identity_readback = pd.read_parquet(STAGED['corpus_embedding_identity'])
question_identity_readback = pd.read_csv(
    STAGED['question_embedding_identity'], dtype=str, keep_default_na=False
)

immutable_hashes_after_scientific_execution = OrderedDict(
    (key, sha256_file(path)) for key, path in AUTHORIZED_INPUT_PATHS.items()
)
if immutable_hashes_after_scientific_execution != immutable_hashes_before:
    raise AssertionError('One or more frozen authorized inputs changed during Cell 7C0.')

scientific_boundary = OrderedDict([
    ('cell_7a3_scores_loaded', False),
    ('answer_key_outcomes_inspected', False),
    ('embedding_model_downloaded', True),
    ('corpus_embeddings_generated', True),
    ('question_embeddings_generated', True),
    ('faiss_index_constructed', True),
    ('semantic_retrieval_executed', True),
    ('common_top20_materialized', True),
    ('quality_scores_loaded', False),
    ('quality_ranks_constructed', False),
    ('quality_reranking_applied', False),
    ('rrf_applied', False),
    ('final_top5_materialized', False),
    ('prompts_materialized', False),
    ('llm_called', False),
    ('adjudication_performed', False),
    ('retrieval_performance_metrics_calculated', False),
    ('rag_performance_metrics_calculated', False),
])

runtime_payload = {
    'python_version': platform.python_version(),
    'python_implementation': platform.python_implementation(),
    'platform': platform.platform(),
    'machine': platform.machine(),
    'processor': platform.processor(),
    'device': DEVICE,
    'cuda_available': bool(torch.cuda.is_available()),
    'cuda_version': torch.version.cuda,
    'cudnn_version': torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'packages': observed_packages,
    'deterministic_controls': {
        'global_seed': FROZEN_SEED,
        'python_hash_seed_environment': os.environ.get('PYTHONHASHSEED'),
        'cublas_workspace_config': os.environ.get('CUBLAS_WORKSPACE_CONFIG'),
        'torch_deterministic_algorithms': bool(torch.are_deterministic_algorithms_enabled()),
        'torch_cudnn_deterministic': bool(torch.backends.cudnn.deterministic),
        'torch_cudnn_benchmark': bool(torch.backends.cudnn.benchmark),
        'tokenizers_parallelism': os.environ.get('TOKENIZERS_PARALLELISM'),
        'corpus_sort': ['packet_id ascending', 'rcv_accession ascending'],
        'question_sort': ['question_id ascending'],
        'retrieval_tie_break': ['semantic score descending', 'rcv_accession ascending', 'packet_id ascending'],
    },
}

timing_payload = {
    'input_load_and_normalization_seconds': float(load_elapsed_seconds),
    'runtime_verification_model_download_and_load_seconds': float(model_load_elapsed_seconds),
    'embedding_generation_seconds_original': float(embedding_generation_seconds),
    'embedding_stage_seconds_current_run': float(embedding_elapsed_seconds),
    'embedding_checkpoint_reused': bool(embedding_checkpoint_reused),
    'faiss_and_retrieval_generation_seconds_original': float(retrieval_generation_seconds),
    'faiss_and_retrieval_stage_seconds_current_run': float(retrieval_elapsed_seconds),
    'retrieval_checkpoint_reused': bool(retrieval_checkpoint_reused),
}

execution_report_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'authorization': {
        'cell_7b5_manifest_path': str(CELL_7B5_PATHS['manifest']),
        'cell_7b5_manifest_sha256': sha256_file(CELL_7B5_PATHS['manifest']),
        'authorization_path': str(CELL_7B5_PATHS['authorization']),
        'authorization_sha256': sha256_file(CELL_7B5_PATHS['authorization']),
        'authorization_decision': EXPECTED_AUTHORIZATION_DECISION,
    },
    'authorized_inputs': {
        key: {'path': str(path), 'sha256': sha256_file(path)}
        for key, path in AUTHORIZED_INPUT_PATHS.items()
    },
    'input_schema': {
        'semantic_corpus_source_columns': semantic_corpus_source_columns,
        'primary_question_source_columns': primary_question_source_columns,
        'semantic_corpus_source_to_canonical_roles': dict(SEMANTIC_CORPUS_SOURCE_TO_CANONICAL),
        'semantic_packet_id_source_column': 'evidence_packet_id',
        'semantic_text_source_column': 'semantic_evidence_text',
        'semantic_packet_id_internal_role': 'packet_id',
        'semantic_text_internal_role': 'semantic_text',
        'question_text_source_column': 'question_text',
    },
    'normalization': {
        'reference_implementation_sha256': sha256_text(NORMALIZATION_REFERENCE_IMPLEMENTATION),
        'corpus_rows_changed_by_normalization': corpus_normalization_changes,
        'question_rows_changed_by_normalization': question_normalization_changes,
    },
    'embedding_execution': {
        'model_id': EXPECTED_MODEL_ID,
        'revision': EXPECTED_MODEL_REVISION,
        'dimension': EXPECTED_DIMENSION,
        'dtype': 'float32',
        'l2_normalized': True,
        'batch_size': BATCH_SIZE,
        'corpus_rows': EXPECTED_CORPUS_ROWS,
        'question_rows': EXPECTED_QUESTION_ROWS,
        'corpus_norm_summary': corpus_norm_summary,
        'question_norm_summary': question_norm_summary,
    },
    'retrieval_execution': {
        'index': 'IndexFlatIP',
        'similarity': 'cosine via inner product of L2-normalized vectors',
        'approximate_search': False,
        'full_exact_score_enumeration_for_global_tie_resolution': True,
        'semantic_candidate_pool_k': EXPECTED_TOP_K,
        'common_pool_for_all_conditions': True,
        'candidate_rows': EXPECTED_CANDIDATE_ROWS,
        'hard_evidence_exclusion': False,
    },
    'runtime': runtime_payload,
    'timing': timing_payload,
    'recovery_checkpoints': {
        'checkpoint_directory': str(CHECKPOINT_DIR),
        'embedding_checkpoint_reused': bool(embedding_checkpoint_reused),
        'retrieval_checkpoint_reused': bool(retrieval_checkpoint_reused),
        'checkpoint_files_are_non_frozen_recovery_artifacts': True,
        'checkpoint_files_included_in_final_manifest': False,
    },
    'scientific_boundary': scientific_boundary,
    'scientific_claim_boundary': (
        'Cell 7C0 materializes frozen embeddings, one exact semantic index, and one common semantic '
        'top-20 candidate pool. It does not establish retrieval quality, answer accuracy, RAG benefit, '
        'clinical validity, clinical utility, or safety.'
    ),
}
stable_write_json(STAGED['execution_report'], execution_report_payload)

prewrite_checks = OrderedDict()
prewrite_checks.update(reverification_checks)
prewrite_checks.update(input_structure_checks)
prewrite_checks.update(embedding_checks)
prewrite_checks.update(retrieval_structure_checks)
prewrite_checks.update(OrderedDict([
    ('model_inventory_exists', STAGED['model_artifact_inventory'].exists()),
    ('model_inventory_nonempty', len(model_files) > 0),
    ('model_inventory_revision_exact', model_inventory_payload['resolved_revision'] == EXPECTED_MODEL_REVISION),
    ('corpus_identity_rows_100920', len(corpus_identity_readback) == EXPECTED_CORPUS_ROWS),
    ('corpus_identity_columns_exact', list(corpus_identity_readback.columns) == [
        'corpus_row_index', 'source_row_index', 'packet_id', 'rcv_accession', 'semantic_text_sha256'
    ]),
    ('corpus_identity_row_index_zero_based', np.array_equal(
        corpus_identity_readback['corpus_row_index'].to_numpy(dtype=np.int64),
        np.arange(EXPECTED_CORPUS_ROWS, dtype=np.int64),
    )),
    ('question_identity_rows_80', len(question_identity_readback) == EXPECTED_QUESTION_ROWS),
    ('question_identity_columns_exact', list(question_identity_readback.columns) == [
        'question_row_index', 'source_row_index', 'question_id', 'question_text_sha256'
    ]),
    ('question_identity_ids_unique', question_identity_readback['question_id'].nunique(dropna=False) == EXPECTED_QUESTION_ROWS),
    ('persisted_index_class_exact', index_readback.__class__.__name__ == 'IndexFlatIP'),
    ('persisted_index_ntotal_exact', int(index_readback.ntotal) == EXPECTED_CORPUS_ROWS),
    ('persisted_index_dimension_exact', int(index_readback.d) == EXPECTED_DIMENSION),
    ('persisted_corpus_embedding_shape_exact', corpus_embeddings_readback.shape == (EXPECTED_CORPUS_ROWS, EXPECTED_DIMENSION)),
    ('persisted_question_embedding_shape_exact', question_embeddings_final.shape == (EXPECTED_QUESTION_ROWS, EXPECTED_DIMENSION)),
    ('persisted_top20_rows_exact', len(semantic_top20_readback) == EXPECTED_CANDIDATE_ROWS),
    ('persisted_top20_schema_exact', list(semantic_top20_readback.columns) == [
        'question_id', 'semantic_rank', 'semantic_score', 'corpus_row_index', 'packet_id', 'rcv_accession'
    ]),
    ('authorized_inputs_immutable', immutable_hashes_after_scientific_execution == immutable_hashes_before),
    ('cell_7a3_scores_remained_unopened', scientific_boundary['cell_7a3_scores_loaded'] is False),
    ('answer_keys_remained_unopened', scientific_boundary['answer_key_outcomes_inspected'] is False),
    ('quality_scores_not_loaded', scientific_boundary['quality_scores_loaded'] is False),
    ('quality_reranking_not_applied', scientific_boundary['quality_reranking_applied'] is False),
    ('rrf_not_applied', scientific_boundary['rrf_applied'] is False),
    ('top5_not_materialized', scientific_boundary['final_top5_materialized'] is False),
    ('prompts_not_materialized', scientific_boundary['prompts_materialized'] is False),
    ('llm_not_called', scientific_boundary['llm_called'] is False),
    ('adjudication_not_performed', scientific_boundary['adjudication_performed'] is False),
    ('rag_metrics_not_calculated', scientific_boundary['rag_performance_metrics_calculated'] is False),
]))

failed_prewrite = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed_prewrite:
    raise RuntimeError('Cell 7C0 prewrite QC failed:\n- ' + '\n- '.join(failed_prewrite))

terminal_decision = (
    'PASS_STAGE7C0_EXACT_PUBMEDBERT_REVISION_CORPUS_AND_QUESTION_EMBEDDINGS_FLOAT32_L2_768_'
    'EXACT_FAISS_INDEXFLATIP_COMMON_SEMANTIC_TOP20_80_QUESTIONS_1600_CANDIDATES_MATERIALIZED_'
    'CHECKSUM_PROTECTED_NO_CELL7A3_SCORES_QUALITY_RERANKING_RRF_TOP5_PROMPTS_LLM_ANSWER_KEY_'
    'OUTCOME_INSPECTION_ADJUDICATION_OR_RAG_METRICS_NEXT_EXECUTION_NOT_AUTHORIZED'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'prewrite_checks': prewrite_checks,
    'passed_checks': int(sum(bool(value) for value in prewrite_checks.values())),
    'failed_checks': int(sum(not bool(value) for value in prewrite_checks.values())),
    'total_checks': len(prewrite_checks),
    'decision': terminal_decision,
    'scientific_boundary': scientific_boundary,
}
stable_write_json(STAGED['qc'], qc_payload)

# Build manifest records for the nine non-manifest outputs from their staged bytes.
non_manifest_output_records = []
for artifact_id, target_path in OUTPUTS.items():
    if artifact_id == 'manifest':
        continue
    staged_path = STAGED[artifact_id]
    if not staged_path.exists():
        raise FileNotFoundError(f'Missing staged Cell 7C0 output: {staged_path}')
    non_manifest_output_records.append({
        'artifact_id': artifact_id,
        'filename': target_path.name,
        'target_path': str(target_path),
        'bytes': int(staged_path.stat().st_size),
        'sha256': sha256_file(staged_path),
        'sidecar_path': str(sidecar_path(target_path)),
    })

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'authorization': {
        'source_cell': '7B5',
        'authorization_decision': EXPECTED_AUTHORIZATION_DECISION,
        'authorization_path': str(CELL_7B5_PATHS['authorization']),
        'authorization_sha256': sha256_file(CELL_7B5_PATHS['authorization']),
        'manifest_path': str(CELL_7B5_PATHS['manifest']),
        'manifest_sha256': sha256_file(CELL_7B5_PATHS['manifest']),
    },
    'upstream_authorized_inputs': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_path': str(sidecar_path(path)),
            'sidecar_valid': True,
        }
        for key, path in AUTHORIZED_INPUT_PATHS.items()
    },
    'execution_design': cell_7b5_authorization['frozen_execution_design'],
    'output_artifacts': non_manifest_output_records,
    'qc': {
        'path': str(OUTPUTS['qc']),
        'sha256': sha256_file(STAGED['qc']),
        'passed_checks': qc_payload['passed_checks'],
        'failed_checks': qc_payload['failed_checks'],
        'total_checks': qc_payload['total_checks'],
    },
    'scientific_boundary': scientific_boundary,
    'terminal_decision': terminal_decision,
    'next_authorized_cell': None,
    'next_required_action': (
        'Create a separate fail-closed authorization that reverifies the complete Cell 7C0 package '
        'before any Cell 7A3 score loading, quality ranking, RRF reranking, or final top-5 context materialization.'
    ),
    'scientific_claim_boundary': execution_report_payload['scientific_claim_boundary'],
}
stable_write_json(STAGED['manifest'], manifest_payload)

print(f'Cell 7C0 staged prewrite QC: {qc_payload["passed_checks"]}/{qc_payload["total_checks"]} PASS')
print('No frozen Drive output has been overwritten.')

Cell 7C0 staged prewrite QC: 94/94 PASS
No frozen Drive output has been overwritten.


## 9. Atomically commit, create SHA-256 sidecars, reverify every output, and print the controlled terminal record

In [10]:
# Commit the nine non-manifest outputs first. The manifest is committed last.
for artifact_id, target_path in OUTPUTS.items():
    if artifact_id == 'manifest':
        continue
    atomic_copy_to_target(STAGED[artifact_id], target_path)
    if sha256_file(target_path) != sha256_file(STAGED[artifact_id]):
        raise AssertionError(f'Atomic copy checksum mismatch for {artifact_id}.')
    write_sidecar(target_path)
    if not sidecar_is_valid(target_path):
        raise AssertionError(f'Fresh output sidecar verification failed for {target_path}.')

atomic_copy_to_target(STAGED['manifest'], OUTPUTS['manifest'])
write_sidecar(OUTPUTS['manifest'])
if not sidecar_is_valid(OUTPUTS['manifest']):
    raise AssertionError('Fresh manifest sidecar verification failed.')

# Fresh full readback from Google Drive.
for artifact_id, target_path in OUTPUTS.items():
    if not target_path.exists():
        raise FileNotFoundError(target_path)
    if not sidecar_is_valid(target_path):
        raise AssertionError(f'Output failed fresh sidecar verification: {target_path}')

readback_manifest = json.loads(OUTPUTS['manifest'].read_text(encoding='utf-8'))
readback_qc = json.loads(OUTPUTS['qc'].read_text(encoding='utf-8'))
readback_report = json.loads(OUTPUTS['execution_report'].read_text(encoding='utf-8'))
readback_model_inventory = json.loads(OUTPUTS['model_artifact_inventory'].read_text(encoding='utf-8'))
readback_corpus_embeddings = np.load(OUTPUTS['corpus_embeddings'], mmap_mode='r')
readback_question_embeddings = np.load(OUTPUTS['question_embeddings'], mmap_mode='r')
readback_index = faiss.read_index(str(OUTPUTS['faiss_index']))
readback_top20 = pd.read_parquet(OUTPUTS['semantic_top20_pool'])
readback_corpus_identity = pd.read_parquet(OUTPUTS['corpus_embedding_identity'])
readback_question_identity = pd.read_csv(
    OUTPUTS['question_embedding_identity'], dtype=str, keep_default_na=False
)

immutable_hashes_final = OrderedDict(
    (key, sha256_file(path)) for key, path in AUTHORIZED_INPUT_PATHS.items()
)

readback_checks = OrderedDict([
    ('ten_outputs_exist', all(path.exists() for path in OUTPUTS.values())),
    ('ten_output_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
    ('manifest_terminal_decision_exact', readback_manifest.get('terminal_decision') == terminal_decision),
    ('manifest_next_authorized_cell_none', readback_manifest.get('next_authorized_cell') is None),
    ('qc_zero_failures', int(readback_qc.get('failed_checks', -1)) == 0),
    ('qc_decision_exact', readback_qc.get('decision') == terminal_decision),
    ('report_authorization_exact', readback_report.get('authorization', {}).get('authorization_decision') == EXPECTED_AUTHORIZATION_DECISION),
    ('model_revision_readback_exact', readback_model_inventory.get('resolved_revision') == EXPECTED_MODEL_REVISION),
    ('corpus_embeddings_readback_shape', readback_corpus_embeddings.shape == (EXPECTED_CORPUS_ROWS, EXPECTED_DIMENSION)),
    ('corpus_embeddings_readback_dtype', readback_corpus_embeddings.dtype == np.dtype('float32')),
    ('question_embeddings_readback_shape', readback_question_embeddings.shape == (EXPECTED_QUESTION_ROWS, EXPECTED_DIMENSION)),
    ('question_embeddings_readback_dtype', readback_question_embeddings.dtype == np.dtype('float32')),
    ('faiss_readback_indexflatip', readback_index.__class__.__name__ == 'IndexFlatIP'),
    ('faiss_readback_ntotal', int(readback_index.ntotal) == EXPECTED_CORPUS_ROWS),
    ('faiss_readback_dimension', int(readback_index.d) == EXPECTED_DIMENSION),
    ('top20_readback_rows_1600', len(readback_top20) == EXPECTED_CANDIDATE_ROWS),
    ('top20_readback_80_questions', readback_top20['question_id'].nunique(dropna=False) == EXPECTED_QUESTION_ROWS),
    ('top20_readback_twenty_each', readback_top20.groupby('question_id', sort=False).size().eq(EXPECTED_TOP_K).all()),
    ('corpus_identity_readback_rows', len(readback_corpus_identity) == EXPECTED_CORPUS_ROWS),
    ('question_identity_readback_rows', len(readback_question_identity) == EXPECTED_QUESTION_ROWS),
    ('authorized_inputs_finally_immutable', immutable_hashes_final == immutable_hashes_before),
    ('cell_7a3_scores_still_unopened', readback_manifest['scientific_boundary']['cell_7a3_scores_loaded'] is False),
    ('quality_reranking_still_false', readback_manifest['scientific_boundary']['quality_reranking_applied'] is False),
    ('rrf_still_false', readback_manifest['scientific_boundary']['rrf_applied'] is False),
    ('top5_still_false', readback_manifest['scientific_boundary']['final_top5_materialized'] is False),
    ('prompts_still_false', readback_manifest['scientific_boundary']['prompts_materialized'] is False),
    ('llm_still_false', readback_manifest['scientific_boundary']['llm_called'] is False),
    ('answer_key_outcomes_still_uninspected', readback_manifest['scientific_boundary']['answer_key_outcomes_inspected'] is False),
    ('adjudication_still_false', readback_manifest['scientific_boundary']['adjudication_performed'] is False),
    ('rag_metrics_still_false', readback_manifest['scientific_boundary']['rag_performance_metrics_calculated'] is False),
])

failed_readback = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_readback:
    raise RuntimeError('Cell 7C0 final readback failed:\n- ' + '\n- '.join(failed_readback))

total_checks = len(prewrite_checks) + len(readback_checks)
passed_checks = total_checks

separator = '=' * 152
print('\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C0')
print('FROZEN EMBEDDING GENERATION, EXACT FAISS INDEXING, AND COMMON SEMANTIC TOP-20 RETRIEVAL EXECUTION')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\nUPSTREAM CELL 7B5 AUTHORIZATION REVERIFICATION')
print(f'Cell 7B5 manifest SHA-256                     : {sha256_file(CELL_7B5_PATHS["manifest"])}')
print('Cell 7B5 terminal PASS verified               : YES')
print('Frozen Cell 7B5 artifacts                     : 4/4 exact hashes + sidecars')
print('Authorized score-blind inputs                 : 6/6 exact hashes + sidecars')
print('Cell 7A3 scores loaded                        : NO')
print('Answer-key outcomes inspected                 : NO')

print('\nEMBEDDING EXECUTION')
print(f'Embedding model                               : {EXPECTED_MODEL_ID}')
print(f'Embedding revision                            : {EXPECTED_MODEL_REVISION}')
print(f'Device                                         : {DEVICE}')
print(f'Corpus embeddings                             : {EXPECTED_CORPUS_ROWS:,} × {EXPECTED_DIMENSION} float32, L2-normalized')
print(f'Question embeddings                           : {EXPECTED_QUESTION_ROWS:,} × {EXPECTED_DIMENSION} float32, L2-normalized')
print(f'Corpus max L2 norm error                      : {corpus_norm_summary["maximum_absolute_norm_error"]:.10g}')
print(f'Question max L2 norm error                    : {question_norm_summary["maximum_absolute_norm_error"]:.10g}')

print('\nEXACT SEMANTIC RETRIEVAL')
print('Similarity/index                              : cosine via exact FAISS IndexFlatIP')
print(f'Indexed corpus vectors                        : {readback_index.ntotal:,}')
print(f'Primary questions                             : {EXPECTED_QUESTION_ROWS}')
print(f'Common semantic candidate pool                : top-{EXPECTED_TOP_K}')
print(f'Candidate rows                                : {len(readback_top20):,}')
print('Global deterministic tie-break                : score desc, RCV asc, packet ID asc')
print(f'Embedding recovery checkpoint reused          : {embedding_checkpoint_reused}')
print(f'Retrieval recovery checkpoint reused          : {retrieval_checkpoint_reused}')
print('Recovery checkpoints in final manifest        : NO')
print('Candidate content displayed                   : NO')

print('\nCELL 7C0 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\nQC checks                                      : {passed_checks}/{total_checks} PASS')

print('\nSCIENTIFIC OPERATIONS')
print('Embeddings generated                          : YES')
print('FAISS index constructed                       : YES')
print('Semantic top-20 retrieval executed            : YES')
print('Cell 7A3 quality scores loaded                : NO')
print('Quality reranking / RRF applied               : NO')
print('Final top-5 contexts materialized             : NO')
print('Prompts materialized                          : NO')
print('LLM called                                     : NO')
print('Answer-key outcomes inspected                 : NO')
print('Adjudication or RAG metrics                    : NO')

print('\nNEXT AUTHORIZATION BOUNDARY')
print('Next execution cell                           : NOT AUTHORIZED')
print('Required next action                          : separate fail-closed authorization before')
print('                                                 Cell 7A3 score loading, quality ranking,')
print('                                                 RRF reranking, or top-5 materialization')
print(f'\nFINAL DECISION                                : {terminal_decision}')
print(separator)


EXPERIMENT 2 — STAGE 7C — CELL 7C0
FROZEN EMBEDDING GENERATION, EXACT FAISS INDEXING, AND COMMON SEMANTIC TOP-20 RETRIEVAL EXECUTION
Notebook                                      : 07_GES_Aware_Genomic_RAG_Cell_7C0_Embedding_and_Semantic_Retrieval_Execution_FINAL_CORRECTED_V6.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM CELL 7B5 AUTHORIZATION REVERIFICATION
Cell 7B5 manifest SHA-256                     : 9c499430815f4a74e364301c0e579d5277d9b62bee1a6b6e76086d59d3366fe9
Cell 7B5 terminal PASS verified               : YES
Frozen Cell 7B5 artifacts                     : 4/4 exact hashes + sidecars
Authorized score-blind inputs                 : 6/6 exact hashes + sidecars
Cell 7A3 scores loaded                        : NO
Answer-key outcomes inspected                 : NO

EMBEDDING EXECUTION
Embedding model                               : NeuML/pubmedbert-base-embeddings
Embedding revision                            : b7952